# 👔 Store Intelligence — Staff vs Customer Classifier

Train a **MobileNetV3-Small** binary classifier to distinguish store staff from customers.
Uses transfer learning from ImageNet with two-phase training (frozen → unfrozen backbone).

**Prerequisites:** Run `build_staff_dataset.py` locally to create the labeled dataset.

**Output:** `training/models/weights/staff_classifier.pth`

In [ ]:
!pip install -q torch torchvision timm matplotlib scikit-learn tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

PROJECT_ROOT = Path('/content/drive/MyDrive/purplle_hackathon')
DATASET_DIR = PROJECT_ROOT / 'training' / 'data' / 'staff_dataset'
WEIGHTS_DIR = PROJECT_ROOT / 'training' / 'models' / 'weights'
os.makedirs(WEIGHTS_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Verify dataset
for split in ['train', 'val']:
    for cls in ['staff', 'customer']:
        p = DATASET_DIR / split / cls
        n = len(list(p.glob('*'))) if p.exists() else 0
        print(f'  {split}/{cls}: {n} images')

In [ ]:
# Define transforms
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = datasets.ImageFolder(str(DATASET_DIR / 'train'), transform=train_transform)
val_ds = datasets.ImageFolder(str(DATASET_DIR / 'val'), transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)

print(f'Classes: {train_ds.classes}')
print(f'Train: {len(train_ds)}, Val: {len(val_ds)}')

In [ ]:
# Show sample images
fig, axes = plt.subplots(2, 8, figsize=(20, 6))
inv_norm = transforms.Normalize(
    mean=[-m/s for m,s in zip(IMAGENET_MEAN, IMAGENET_STD)],
    std=[1/s for s in IMAGENET_STD]
)
for i, ax in enumerate(axes.flat):
    if i < len(val_ds):
        img, label = val_ds[i]
        img = inv_norm(img).permute(1,2,0).clamp(0,1).numpy()
        ax.imshow(img)
        ax.set_title(train_ds.classes[label], fontsize=9)
    ax.axis('off')
plt.suptitle('Sample Images', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Create model
model = timm.create_model('mobilenetv3_small_100', pretrained=True, num_classes=2)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params:,}')
print(f'Trainable: {trainable:,}')

In [ ]:
from tqdm import tqdm

def train_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=device.type=='cuda'):
            out = model(imgs)
            loss = criterion(out, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * imgs.size(0)
        correct += (out.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return total_loss/total, correct/total

def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs)
            loss = criterion(out, labels)
            total_loss += loss.item() * imgs.size(0)
            correct += (out.argmax(1) == labels).sum().item()
            total += imgs.size(0)
    return total_loss/total, correct/total

print('Training functions defined')

In [ ]:
# Phase 1: Freeze backbone, train classifier head
FREEZE_EPOCHS = 10
TOTAL_EPOCHS = 30

# Freeze all except classifier
for name, p in model.named_parameters():
    if 'classifier' not in name:
        p.requires_grad = False

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, TOTAL_EPOCHS)
scaler = torch.cuda.amp.GradScaler(enabled=device.type=='cuda')

history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}

print(f'Phase 1: Training classifier head ({FREEZE_EPOCHS} epochs)...')
for epoch in range(FREEZE_EPOCHS):
    tl, ta = train_epoch(model, train_loader, criterion, optimizer, scaler, device)
    vl, va = eval_epoch(model, val_loader, criterion, device)
    scheduler.step()
    history['train_loss'].append(tl); history['val_loss'].append(vl)
    history['train_acc'].append(ta); history['val_acc'].append(va)
    print(f'  Epoch {epoch+1}/{FREEZE_EPOCHS} — train_loss={tl:.4f} train_acc={ta:.3f} val_loss={vl:.4f} val_acc={va:.3f}')

# Phase 2: Unfreeze all, fine-tune
print(f'\nPhase 2: Fine-tuning all layers ({TOTAL_EPOCHS-FREEZE_EPOCHS} epochs)...')
for p in model.parameters():
    p.requires_grad = True
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, TOTAL_EPOCHS - FREEZE_EPOCHS)

for epoch in range(FREEZE_EPOCHS, TOTAL_EPOCHS):
    tl, ta = train_epoch(model, train_loader, criterion, optimizer, scaler, device)
    vl, va = eval_epoch(model, val_loader, criterion, device)
    scheduler.step()
    history['train_loss'].append(tl); history['val_loss'].append(vl)
    history['train_acc'].append(ta); history['val_acc'].append(va)
    print(f'  Epoch {epoch+1}/{TOTAL_EPOCHS} — train_loss={tl:.4f} train_acc={ta:.3f} val_loss={vl:.4f} val_acc={va:.3f}')

print('\nTraining complete!')

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history['train_loss'], label='Train'); ax1.plot(history['val_loss'], label='Val')
ax1.set_title('Loss'); ax1.legend(); ax1.grid(True, alpha=0.3)
ax1.axvline(FREEZE_EPOCHS-0.5, color='r', linestyle='--', alpha=0.5, label='Unfreeze')
ax2.plot(history['train_acc'], label='Train'); ax2.plot(history['val_acc'], label='Val')
ax2.set_title('Accuracy'); ax2.legend(); ax2.grid(True, alpha=0.3)
ax2.axvline(FREEZE_EPOCHS-0.5, color='r', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate with confusion matrix
from sklearn.metrics import confusion_matrix, classification_report

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        out = model(imgs.to(device))
        preds = out.argmax(1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=train_ds.classes))
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(6,5))
ax.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i,j]), ha='center', va='center', fontsize=16)
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(train_ds.classes); ax.set_yticklabels(train_ds.classes)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Save model
save_path = WEIGHTS_DIR / 'staff_classifier.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'classes': train_ds.classes,
    'accuracy': history['val_acc'][-1],
}, str(save_path))
print(f'Model saved to: {save_path}')
print(f'Final val accuracy: {history["val_acc"][-1]:.3f}')

## ✅ Summary

- MobileNetV3-Small trained with transfer learning for staff/customer classification
- Model saved to `training/models/weights/staff_classifier.pth`

**Tip:** If staff uniforms differ significantly between stores, train separate
per-store models for better accuracy.

**Next:** Run `03_reid_embedding.ipynb` to train Re-ID embeddings.